# 📖 Lab 4: Ticket Reservation (Deep Dive)

**Non-functional requirement:** *The system should prioritize consistency for booking events (no double booking).*

In Lab 3 we prevented double bookings with `SELECT ... FOR UPDATE`. But we left a major UX problem: users fill out payment details only to discover the ticket is gone. This deep dive explores **4 approaches** to temporary ticket reservations — from bad to great.

## 🏗️ Architecture — Before (Lab 3)

```
┌────────┐       ┌─────────────┐       ┌─────────────────┐       ┌──────────────┐
│ Client │──────>│ API Gateway │──────>│ Booking Service  │──TX──>│  PostgreSQL  │
└────────┘       └─────────────┘       │                 │       └──────────────┘
                                       │  book(tickets)   │
                  POST /bookings        │       │          │
                                       │       v          │
                                       │    Stripe        │
                                       └─────────────────┘

Problem: User fills payment form → ticket already taken → 😡
```

## 🏗️ Architecture — After (Redis Distributed Lock)

```
┌────────┐       ┌─────────────┐       ┌─────────────────┐       ┌──────────────┐
│ Client │──────>│ API Gateway │──────>│ Booking Service  │──────>│  PostgreSQL  │
└────────┘       └─────────────┘       │                 │       │  tickets     │
                                       │  reserve ──────>│──────>│  bookings    │
               POST /bookings/reserve   │                 │       └──────────────┘
               POST /bookings/confirm   │  confirm ──────>│
                                       │       │         │       ┌──────────────┐
                                       │       v         │──────>│    Redis     │
                                       │    Stripe       │       │  Ticket Lock │
                                       └─────────────────┘       │  {id: user}  │
                                                                  │  TTL 10 min  │
                                                                  └──────────────┘
```

Two-phase booking: **reserve** (Redis lock) → **confirm** (payment + DB update). Ticket only has 2 DB states: `available` / `booked`. Reservation lives in Redis with automatic TTL expiration.

## The 4 Approaches

| # | Approach | Verdict |
|---|----------|---------|
| 1 | Long-running DB locks (`SELECT FOR UPDATE` held for minutes) | ❌ Bad |
| 2 | Status + expiration column with cron job cleanup | 🟡 Good |
| 3 | Inline expiration check (no cron needed) | 🟢 Great |
| 4 | Redis distributed lock with TTL | 🟢 Great |

## Learning Objectives

- See why long-running DB locks fail under load
- Build the cron-based approach and observe the expiration delay problem
- Implement inline expiration that's self-healing with no cron
- Build the Redis distributed lock approach with two-phase booking (reserve → confirm)
- Handle edge cases: TTL expiry, concurrent reservations, multi-ticket locking
- See the seat map reflect reserved tickets in real-time

## 🛠️ Setup

Make sure both PostgreSQL and Redis are running:

```bash
cd system-designs/ticketmaster
docker-compose up -d
pip install redis   # if not already installed
```

Select the **"Ticketmaster (Python)"** kernel.

**Optional:** Open [RedisInsight](http://localhost:5540) to watch locks in real-time.  
Add a new connection: Host `localhost`, Port `6380`.

In [1]:
import psycopg2
import psycopg2.extras
import redis
import threading
import time
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

redis_client = redis.Redis(host="localhost", port=6380, decode_responses=True)

# Test both connections
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM tickets WHERE status = 'available'")
pg_count = cur.fetchone()[0]
cur.close()
conn.close()

redis_pong = redis_client.ping()

print(f"✅ PostgreSQL: {pg_count} available tickets")
print(f"✅ Redis: {'connected' if redis_pong else 'FAILED'}")

✅ PostgreSQL: 497 available tickets
✅ Redis: connected


## ❌ Bad Approach: Long-Running Database Locks

The first thing many candidates propose: just keep a `SELECT ... FOR UPDATE` transaction open while the user fills out their payment form. The row stays locked until the transaction commits (user pays) or rolls back (user abandons).

```
Time    User A                                      Database
────    ──────                                      ────────
  0     BEGIN TRANSACTION
  1     SELECT ... FOR UPDATE on ticket 42          Row locked 🔒
  2     (user is typing credit card info...)        Row still locked 🔒
  3     ...5 minutes pass...                        Row STILL locked 🔒 😬
  4     User submits payment → COMMIT               Row unlocked 🔓
```

### Why This Is Bad

| Problem | Impact |
|---------|--------|
| **Connection held for minutes** | Each open transaction holds a DB connection. With 10,000 concurrent users, you need 10,000 connections. PostgreSQL's default max is 100. |
| **Lock contention** | Other transactions wanting these rows **block and wait**. Under load, this cascades into timeouts. |
| **Deadlock risk** | If User A locks ticket 1 then tries ticket 2, while User B locks ticket 2 then tries ticket 1 → deadlock. |
| **Abandoned sessions** | If the user closes their browser, the connection may hang until TCP timeout (could be minutes). |
| **No graceful expiration** | PostgreSQL's `lock_timeout` kills the transaction with an error, not a graceful "your time expired" message. |

Let's see this in action — and watch it fail.

In [ ]:
# ❌ BAD: Long-running database lock
# Simulates holding a SELECT FOR UPDATE for a long time

def reserve_with_db_lock(user_id: int, ticket_id: int, hold_seconds: int = 5):
    """
    Bad approach: hold a database transaction open while user "pays".
    The row is locked for the entire duration.
    """
    conn = get_connection()
    conn.autocommit = False
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        cur.execute("""
            SELECT id, status, price FROM tickets
            WHERE id = %s
            FOR UPDATE
        """, (ticket_id,))
        ticket = cur.fetchone()

        if ticket["status"] != "available":
            conn.rollback()
            return {"success": False, "error": f"Ticket already {ticket['status']}"}

        print(f"  🔒 User {user_id}: locked ticket {ticket_id}, holding for {hold_seconds}s...")
        print(f"     (simulating user typing credit card info)")

        # This is the problem — the connection and lock are held open
        # while we wait for the user to complete payment
        time.sleep(hold_seconds)

        cur.execute("UPDATE tickets SET status = 'booked' WHERE id = %s", (ticket_id,))
        conn.commit()
        print(f"  ✅ User {user_id}: committed! Ticket booked.")
        return {"success": True}

    except Exception as e:
        conn.rollback()
        print(f"  ❌ User {user_id}: {e}")
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()

# Pick a ticket for the demo
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1")
bad_demo_ticket = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Ticket {bad_demo_ticket}: two users will race with long-running DB locks\n")

# User A holds the lock for 3 seconds, User B tries immediately
results = {}

def user_a_slow():
    results["A"] = reserve_with_db_lock(user_id=1, ticket_id=bad_demo_ticket, hold_seconds=3)

def user_b_blocked():
    time.sleep(0.2)  # Start slightly after A
    print(f"  ⏳ User B: trying to lock ticket {bad_demo_ticket}...")
    start = time.time()
    results["B"] = reserve_with_db_lock(user_id=2, ticket_id=bad_demo_ticket, hold_seconds=1)
    wait_time = time.time() - start
    print(f"  ⏱️  User B waited {wait_time:.1f}s total (blocked on lock)")

t1 = threading.Thread(target=user_a_slow)
t2 = threading.Thread(target=user_b_blocked)
t1.start(); t2.start()
t1.join(); t2.join()

print(f"\n📊 Results:")
print(f"  User A: {'✅' if results['A']['success'] else '❌'}")
print(f"  User B: {'✅' if results['B']['success'] else '❌'} — {results['B'].get('error', 'booked')}")

print(f"\n⚠️  User B was BLOCKED for ~3 seconds waiting for User A's lock.")
print(f"   Now imagine 10,000 users all blocked like this. The DB would melt.")

# Reset
conn = get_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available' WHERE id = %s", (bad_demo_ticket,))
conn.commit()
cur.close()
conn.close()

## 🟡 Good Approach: Status + Expiration Column with Cron Job

Better idea: add a `reserved` status and a `reserved_until` timestamp to the ticket row. Use **short** transactions to reserve (set status and expiration), and a **cron job** to periodically clean up expired reservations.

```sql
-- The ticket can be in 3 states:
-- available → ready to reserve
-- reserved  → someone is paying (check reserved_until for expiry)
-- booked    → sold, done

ALTER TABLE tickets ADD COLUMN reserved_until TIMESTAMP;
ALTER TABLE tickets ADD COLUMN reserved_by INTEGER;
```

The cron job runs every N seconds and resets expired reservations:
```sql
UPDATE tickets SET status = 'available', reserved_until = NULL, reserved_by = NULL
WHERE status = 'reserved' AND reserved_until < NOW();
```

### Why This Is Better (But Not Great)

| ✅ Wins | ❌ Problems |
|---------|-----------|
| Short transactions — no long-held locks | **Cron delay**: if cron runs every 30s, tickets can be "stuck" as reserved for up to 30s after expiry |
| Proper status tracking in DB | **Cron reliability**: if the cron job fails or is delayed, tickets stay reserved indefinitely |
| Other services can see the status | For hot events (10M users), even a few seconds of delay means lost sales |

In [ ]:
# 🟡 GOOD: Status + expiration column approach
# First, add the columns (idempotent — skips if already exist)

conn = get_connection()
cur = conn.cursor()
cur.execute("""
    DO $$ BEGIN
        ALTER TABLE tickets ADD COLUMN reserved_until TIMESTAMP;
    EXCEPTION WHEN duplicate_column THEN NULL;
    END $$;
""")
cur.execute("""
    DO $$ BEGIN
        ALTER TABLE tickets ADD COLUMN reserved_by INTEGER;
    EXCEPTION WHEN duplicate_column THEN NULL;
    END $$;
""")
conn.commit()
cur.close()
conn.close()

CRON_RESERVATION_TTL = 10  # 10 seconds for demo

def reserve_with_status_column(user_id: int, ticket_id: int) -> dict:
    """
    Good approach: short transaction to set status='reserved' + reserved_until.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    try:
        cur.execute("""
            UPDATE tickets
            SET status = 'reserved',
                reserved_until = NOW() + interval '%s seconds',
                reserved_by = %s
            WHERE id = %s AND status = 'available'
            RETURNING id
        """, (CRON_RESERVATION_TTL, user_id, ticket_id))

        result = cur.fetchone()
        conn.commit()

        if result:
            return {"success": True, "message": f"Reserved for {CRON_RESERVATION_TTL}s"}
        else:
            return {"success": False, "error": "Ticket not available"}
    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()


def cron_cleanup_expired():
    """Simulates the cron job that resets expired reservations."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        UPDATE tickets
        SET status = 'available', reserved_until = NULL, reserved_by = NULL
        WHERE status = 'reserved' AND reserved_until < NOW()
    """)
    cleaned = cur.rowcount
    conn.commit()
    cur.close()
    conn.close()
    return cleaned


# Demo: reserve, wait for expiry, run cron
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1")
cron_ticket_id = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Reserving ticket {cron_ticket_id} with status column approach\n")

result = reserve_with_status_column(user_id=1, ticket_id=cron_ticket_id)
print(f"📌 Reserve result: {result}")

# Check status
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT status, reserved_until, reserved_by FROM tickets WHERE id = %s", (cron_ticket_id,))
t = cur.fetchone()
print(f"📊 Ticket state: status={t['status']}, reserved_until={t['reserved_until']}, reserved_by={t['reserved_by']}")
cur.close()
conn.close()

# User B tries — blocked
result_b = reserve_with_status_column(user_id=2, ticket_id=cron_ticket_id)
print(f"\n🙅 User B tries: {result_b}")

# Wait for expiry
print(f"\n⏳ Waiting {CRON_RESERVATION_TTL}s for reservation to expire...")
time.sleep(CRON_RESERVATION_TTL + 1)

# Without cron, the ticket is STILL marked as reserved!
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT status, reserved_until FROM tickets WHERE id = %s", (cron_ticket_id,))
t = cur.fetchone()
print(f"\n⚠️  After expiry, ticket is STILL: status={t['status']} (reserved_until={t['reserved_until']})")
print(f"   The DB doesn't know the reservation expired! We need the cron to clean it up.")
cur.close()
conn.close()

# Run cron
cleaned = cron_cleanup_expired()
print(f"\n🧹 Cron job ran: cleaned up {cleaned} expired reservation(s)")

conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT status FROM tickets WHERE id = %s", (cron_ticket_id,))
print(f"📊 Ticket {cron_ticket_id} is now: {cur.fetchone()['status']}")
cur.close()
conn.close()

print(f"\n💡 The problem: between expiry and cron, the ticket was unavailable.")
print(f"   For a hot event, even seconds of delay = lost revenue.")

## 🟢 Great Approach 1: Inline Expiration Check (No Cron Needed)

Key insight: the status of a ticket is the **combination** of two things:
1. Is it `available`?
2. Is it `reserved` but the reservation has **expired**?

Instead of waiting for a cron job, we check the expiration **inline** in every transaction. When reserving, we look for tickets that are either `AVAILABLE` or `RESERVED with an expired timestamp`.

```sql
-- Reserve: claim ticket if available OR if previous reservation expired
UPDATE tickets
SET status = 'reserved', reserved_until = NOW() + interval '10 minutes', reserved_by = :userId
WHERE id = :ticketId
  AND (status = 'available' OR (status = 'reserved' AND reserved_until < NOW()))
```

### Why This Is Great

| ✅ Win | Details |
|-------|---------|
| **No cron dependency** | Expired reservations are reclaimed on-demand — zero delay |
| **Self-healing** | Even if no cleanup job ever runs, the system still works correctly |
| **Simpler ops** | No background process to monitor, no failure modes from cron |

### Trade-offs

| ❌ Trade-off | Details |
|-------------|---------|
| **Reads are slightly slower** | Every read must check `reserved_until` — solvable with compound indexes |
| **Dirty data in DB** | Some rows say "reserved" but are actually expired — less legible for debugging. A sweep job can clean this up, but the system doesn't depend on it |

In [ ]:
# 🟢 GREAT 1: Inline expiration check — no cron needed

INLINE_TTL = 5  # 5 seconds for demo

def reserve_inline(user_id: int, ticket_id: int) -> dict:
    """
    Great approach: reserve if available OR if previous reservation expired.
    No cron needed — expired reservations are reclaimed on-demand.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    try:
        # The key trick: check for available OR expired reservation
        cur.execute("""
            UPDATE tickets
            SET status = 'reserved',
                reserved_until = NOW() + interval '%s seconds',
                reserved_by = %s
            WHERE id = %s
              AND (status = 'available' OR (status = 'reserved' AND reserved_until < NOW()))
            RETURNING id
        """, (INLINE_TTL, user_id, ticket_id))

        result = cur.fetchone()
        conn.commit()

        if result:
            return {"success": True, "message": f"Reserved for {INLINE_TTL}s", "userId": user_id}
        else:
            return {"success": False, "error": "Ticket not available (reserved by another user or already booked)"}
    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()


# Demo: reserve, let it expire, then another user grabs it — no cron!
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1")
inline_ticket = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🎯 Ticket {inline_ticket}: demonstrating inline expiration\n")

# User A reserves
result_a = reserve_inline(user_id=1, ticket_id=inline_ticket)
print(f"📌 User A reserves: {result_a}")

# User B tries immediately — blocked
result_b = reserve_inline(user_id=2, ticket_id=inline_ticket)
print(f"🙅 User B tries now: {result_b}")

# Wait for User A's reservation to expire
print(f"\n⏳ Waiting {INLINE_TTL + 1}s for User A's reservation to expire...")
time.sleep(INLINE_TTL + 1)

# User B tries again — succeeds WITHOUT any cron job!
result_b2 = reserve_inline(user_id=2, ticket_id=inline_ticket)
print(f"\n🎉 User B tries again: {result_b2}")

if result_b2["success"]:
    print(f"\n✅ User B grabbed the ticket immediately after expiry!")
    print(f"   No cron job ran. The inline check handled it.")

# Reset
conn = get_connection()
cur = conn.cursor()
cur.execute("UPDATE tickets SET status = 'available', reserved_until = NULL, reserved_by = NULL WHERE id = %s", (inline_ticket,))
conn.commit()
cur.close()
conn.close()

## 🟢 Great Approach 2: Redis Distributed Lock with TTL

If PostgreSQL already handles inline expiration, why bring in Redis at all?

**Because PostgreSQL doesn't natively support row-level TTLs.** The inline approach works, but:
- Every reservation query must check `reserved_until < NOW()` — extra filter on every read
- The ticket table is "dirty" — some rows say `reserved` but are actually expired
- Under extreme load (10M users, one event), the DB is doing both locking AND expiration checks

Redis gives us:
- **Automatic key expiration** — built-in TTL, no filtering needed
- **In-memory speed** — lock acquisition in microseconds, not milliseconds
- **Separation of concerns** — ephemeral state (reservations) in Redis, durable state (bookings) in PostgreSQL

The key Redis command:

```
SET ticket:{id} {userId} NX EX 600
```

- `NX` — **Only set if the key does Not eXist** (atomic — no race condition)
- `EX 600` — **Expire after 600 seconds** (10 minutes)
- Returns `True` if the lock was acquired, `None` if someone else holds it

This is atomic: if two users try to lock the same ticket at the exact same millisecond, only one succeeds. No race condition possible.

Let's see it in action.

In [2]:
# Demo: Redis SET NX EX — the atomic lock

# Clean slate
redis_client.delete("ticket:demo")

# User A tries to reserve
locked_a = redis_client.set("ticket:demo", "user_A", nx=True, ex=30)
print(f"User A tries to lock: {'✅ Acquired!' if locked_a else '❌ Failed'}")
print(f"  Key value: {redis_client.get('ticket:demo')}")
print(f"  TTL remaining: {redis_client.ttl('ticket:demo')}s")

# User B tries to reserve the SAME ticket
locked_b = redis_client.set("ticket:demo", "user_B", nx=True, ex=30)
print(f"\nUser B tries to lock: {'✅ Acquired!' if locked_b else '❌ Failed — ticket already reserved!'}")
print(f"  Key still belongs to: {redis_client.get('ticket:demo')}")

# Cleanup
redis_client.delete("ticket:demo")
print("\n🧹 Cleaned up demo key")

User A tries to lock: ✅ Acquired!
  Key value: user_A
  TTL remaining: 30s

User B tries to lock: ❌ Failed — ticket already reserved!
  Key still belongs to: user_A

🧹 Cleaned up demo key


## 🔧 Building the Two-Phase Booking Service

Now let's build the real thing. Our booking flow splits into two API endpoints:

### Phase 1: `POST /bookings/reserve`
- Acquire Redis lock on each ticket (`SET ticket:{id} userId NX EX 600`)
- Track reserved tickets in a Redis Set per event (`event:{eventId}:reserved`)
- Create a Booking record in PostgreSQL with status `in-progress`
- Return `bookingId` + TTL to the client (client shows countdown timer)

### Phase 2: `POST /bookings/confirm`
- Verify the Redis lock is still held by this user (TTL hasn't expired)
- Process payment (simulated — in production this is Stripe)
- Update ticket status to `booked` and booking status to `confirmed` in PostgreSQL
- Release the Redis lock

In [3]:
RESERVATION_TTL = 30  # 30 seconds for demo (10 minutes in production)

def reserve_tickets(user_id: int, event_id: int, ticket_ids: list[int]) -> dict:
    """
    Phase 1: POST /bookings/reserve
    
    Acquires Redis locks on tickets, creates an in-progress booking.
    If any ticket is already reserved or booked, rolls everything back.
    """
    locked_keys = []

    try:
        # Step 1: Try to lock each ticket in Redis (atomic per key)
        for tid in ticket_ids:
            lock_key = f"ticket:{tid}"
            acquired = redis_client.set(lock_key, str(user_id), nx=True, ex=RESERVATION_TTL)

            if not acquired:
                # Someone else holds this ticket — release any locks we already got
                for key in locked_keys:
                    redis_client.delete(key)
                holder = redis_client.get(lock_key)
                return {
                    "success": False,
                    "error": f"Ticket {tid} is already reserved by another user",
                }

            locked_keys.append(lock_key)

        # Step 2: Track reserved tickets per event (for seat map queries)
        reserved_set_key = f"event:{event_id}:reserved"
        for tid in ticket_ids:
            redis_client.sadd(reserved_set_key, str(tid))

        # Step 3: Verify tickets are available in the database
        conn = get_connection()
        cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

        cur.execute("""
            SELECT id, status, price FROM tickets
            WHERE id = ANY(%s)
            FOR UPDATE
        """, (ticket_ids,))
        tickets = cur.fetchall()

        for t in tickets:
            if t["status"] != "available":
                conn.rollback()
                # Release Redis locks
                for key in locked_keys:
                    redis_client.delete(key)
                for tid in ticket_ids:
                    redis_client.srem(reserved_set_key, str(tid))
                return {"success": False, "error": f"Ticket {t['id']} is already {t['status']} in DB"}

        # Step 4: Create booking with status "in-progress"
        total_price = sum(float(t["price"]) for t in tickets)
        cur.execute("""
            INSERT INTO bookings (user_id, event_id, total_price, status)
            VALUES (%s, %s, %s, 'in-progress')
            RETURNING id
        """, (user_id, event_id, total_price))
        booking_id = cur.fetchone()["id"]

        # Link tickets to booking
        for tid in ticket_ids:
            cur.execute("INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s)", (booking_id, tid))

        conn.commit()
        cur.close()
        conn.close()

        return {
            "success": True,
            "bookingId": booking_id,
            "totalPrice": total_price,
            "ttlSeconds": RESERVATION_TTL,
            "message": f"Tickets reserved! Complete payment within {RESERVATION_TTL}s.",
        }

    except Exception as e:
        for key in locked_keys:
            redis_client.delete(key)
        return {"success": False, "error": str(e)}


def confirm_booking(user_id: int, booking_id: int) -> dict:
    """
    Phase 2: POST /bookings/confirm
    
    Verifies the user still holds the reservation, processes payment,
    updates DB to booked/confirmed, and releases Redis locks.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Step 1: Fetch the booking and its tickets
        cur.execute("""
            SELECT b.id, b.user_id, b.event_id, b.status, b.total_price
            FROM bookings b
            WHERE b.id = %s
        """, (booking_id,))
        booking = cur.fetchone()

        if not booking:
            return {"success": False, "error": "Booking not found"}
        if booking["user_id"] != user_id:
            return {"success": False, "error": "This booking belongs to another user"}
        if booking["status"] != "in-progress":
            return {"success": False, "error": f"Booking is already {booking['status']}"}

        cur.execute("SELECT ticket_id FROM booking_tickets WHERE booking_id = %s", (booking_id,))
        ticket_ids = [row["ticket_id"] for row in cur.fetchall()]

        # Step 2: Verify Redis locks are still held by this user
        for tid in ticket_ids:
            lock_holder = redis_client.get(f"ticket:{tid}")
            if lock_holder != str(user_id):
                conn.rollback()
                return {
                    "success": False,
                    "error": f"Reservation expired for ticket {tid}! Someone else may have grabbed it.",
                }

        # Step 3: Process payment (simulated — in production: Stripe PaymentIntent)
        time.sleep(0.1)  # Simulating payment processing
        payment_success = True  # In production: check Stripe webhook

        if not payment_success:
            conn.rollback()
            return {"success": False, "error": "Payment failed"}

        # Step 4: Finalize in database (within a transaction)
        cur.execute("""
            UPDATE tickets SET status = 'booked'
            WHERE id = ANY(%s)
        """, (ticket_ids,))

        cur.execute("""
            UPDATE bookings SET status = 'confirmed'
            WHERE id = %s
        """, (booking_id,))

        conn.commit()

        # Step 5: Release Redis locks and remove from reserved set
        reserved_set_key = f"event:{booking['event_id']}:reserved"
        for tid in ticket_ids:
            redis_client.delete(f"ticket:{tid}")
            redis_client.srem(reserved_set_key, str(tid))

        return {
            "success": True,
            "bookingId": booking_id,
            "totalPrice": float(booking["total_price"]),
            "message": "Payment confirmed! Tickets are yours. 🎉",
        }

    except Exception as e:
        conn.rollback()
        return {"success": False, "error": str(e)}
    finally:
        cur.close()
        conn.close()

print("✅ reserve_tickets() and confirm_booking() defined.")

✅ reserve_tickets() and confirm_booking() defined.


## 🧪 Test 1: The Happy Path

One user reserves tickets, then confirms payment. The full flow.

In [4]:
# Pick 2 available tickets
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id, section, row_label, seat_number, price
    FROM tickets WHERE event_id = 1 AND status = 'available'
    ORDER BY id LIMIT 2
""")
test_tickets = cur.fetchall()
cur.close()
conn.close()

test_ids = [t["id"] for t in test_tickets]
print("🎫 Selected tickets:")
for t in test_tickets:
    print(f"   Ticket {t['id']}: {t['section']}-{t['row_label']}-Seat {t['seat_number']} (${t['price']})")

# Phase 1: Reserve
print(f"\n{'='*60}")
print("📌 Phase 1: RESERVE")
print(f"{'='*60}")
reserve_result = reserve_tickets(user_id=1, event_id=1, ticket_ids=test_ids)
print(f"\n{reserve_result}")

# Check Redis state
print(f"\n🔍 Redis state after reservation:")
for tid in test_ids:
    holder = redis_client.get(f"ticket:{tid}")
    ttl = redis_client.ttl(f"ticket:{tid}")
    print(f"   ticket:{tid} → holder={holder}, TTL={ttl}s")

reserved = redis_client.smembers("event:1:reserved")
print(f"   event:1:reserved → {reserved}")

# Phase 2: Confirm (simulating user completing payment)
print(f"\n{'='*60}")
print("💳 Phase 2: CONFIRM (payment)")
print(f"{'='*60}")
confirm_result = confirm_booking(user_id=1, booking_id=reserve_result["bookingId"])
print(f"\n{confirm_result}")

# Verify final state
print(f"\n🔍 Final state:")
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, status FROM tickets WHERE id = ANY(%s)", (test_ids,))
for t in cur.fetchall():
    print(f"   Ticket {t['id']}: {t['status']}")
cur.execute("SELECT id, status FROM bookings WHERE id = %s", (reserve_result["bookingId"],))
b = cur.fetchone()
print(f"   Booking {b['id']}: {b['status']}")
cur.close()
conn.close()

# Redis locks should be gone
for tid in test_ids:
    print(f"   Redis ticket:{tid} → {redis_client.get(f'ticket:{tid}') or '(released)'}")

🎫 Selected tickets:
   Ticket 2: FLOOR-A-Seat 2 ($250.00)
   Ticket 3: FLOOR-A-Seat 3 ($250.00)

📌 Phase 1: RESERVE

{'success': True, 'bookingId': 2, 'totalPrice': 500.0, 'ttlSeconds': 30, 'message': 'Tickets reserved! Complete payment within 30s.'}

🔍 Redis state after reservation:
   ticket:2 → holder=1, TTL=30s
   ticket:3 → holder=1, TTL=30s
   event:1:reserved → {'3', '2'}

💳 Phase 2: CONFIRM (payment)

{'success': True, 'bookingId': 2, 'totalPrice': 500.0, 'message': 'Payment confirmed! Tickets are yours. 🎉'}

🔍 Final state:
   Ticket 2: booked
   Ticket 3: booked
   Booking 2: confirmed
   Redis ticket:2 → (released)
   Redis ticket:3 → (released)


## 🧪 Test 2: Concurrent Reservation — User B Gets Blocked Immediately

The key UX improvement: User B finds out **instantly** that the ticket is reserved, instead of after filling out a payment form.

In [5]:
# Pick a fresh available ticket
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT id FROM tickets WHERE event_id = 1 AND status = 'available'
    ORDER BY id LIMIT 1
""")
contested_id = cur.fetchone()["id"]
cur.close()
conn.close()

print(f"🎯 Both users want ticket {contested_id}\n")

# User A reserves
results = {}

def user_reserves(user_id, event_id, ticket_ids):
    results[user_id] = reserve_tickets(user_id=user_id, event_id=event_id, ticket_ids=ticket_ids)

thread_a = threading.Thread(target=user_reserves, args=(10, 1, [contested_id]))
thread_b = threading.Thread(target=user_reserves, args=(20, 1, [contested_id]))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("Results:")
for uid, result in sorted(results.items()):
    status = "✅ RESERVED" if result["success"] else "❌ BLOCKED"
    print(f"  User {uid}: {status}")
    if result["success"]:
        print(f"    → bookingId={result['bookingId']}, TTL={result['ttlSeconds']}s")
    else:
        print(f"    → {result['error']}")

print(f"\n💡 User B knows IMMEDIATELY the ticket is taken.")
print(f"   No wasted time filling out payment forms!")

# Cleanup: release the lock and booking
redis_client.delete(f"ticket:{contested_id}")
redis_client.srem("event:1:reserved", str(contested_id))
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM booking_tickets WHERE ticket_id = %s", (contested_id,))
cur.execute("DELETE FROM bookings WHERE status = 'in-progress'")
conn.commit()
cur.close()
conn.close()

🎯 Both users want ticket 4

Results:
  User 10: ✅ RESERVED
    → bookingId=3, TTL=30s
  User 20: ❌ BLOCKED
    → Ticket 4 is already reserved by another user

💡 User B knows IMMEDIATELY the ticket is taken.
   No wasted time filling out payment forms!


## ⏰ Test 3: TTL Expiration — Abandoned Reservation

What if User A reserves but never pays? The Redis lock auto-expires and the ticket becomes available again. No cron job needed!

We're using a 30-second TTL for this demo (production would be 10 minutes).

In [6]:
# Use a short TTL for this demo
SHORT_TTL = 5  # 5 seconds

conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 1")
expire_ticket_id = cur.fetchone()["id"]
cur.close()
conn.close()

# Manually acquire lock with short TTL
redis_client.set(f"ticket:{expire_ticket_id}", "user_lazy", nx=True, ex=SHORT_TTL)
redis_client.sadd("event:1:reserved", str(expire_ticket_id))

print(f"📌 User 'lazy' reserved ticket {expire_ticket_id} with {SHORT_TTL}s TTL\n")

# Watch the TTL count down
for i in range(SHORT_TTL + 2):
    ttl = redis_client.ttl(f"ticket:{expire_ticket_id}")
    holder = redis_client.get(f"ticket:{expire_ticket_id}")
    if holder:
        print(f"  ⏰ t={i}s — holder={holder}, TTL={ttl}s")
    else:
        print(f"  ⏰ t={i}s — 🔓 LOCK EXPIRED! Ticket is available again!")
        # Clean up the reserved set (in production, a periodic sweep handles this)
        redis_client.srem("event:1:reserved", str(expire_ticket_id))
        break
    time.sleep(1)

# Now another user CAN reserve it
can_lock = redis_client.set(f"ticket:{expire_ticket_id}", "user_fast", nx=True, ex=30)
print(f"\n  User 'fast' tries to reserve: {'✅ Got it!' if can_lock else '❌ Still locked'}")
redis_client.delete(f"ticket:{expire_ticket_id}")
redis_client.srem("event:1:reserved", str(expire_ticket_id))

📌 User 'lazy' reserved ticket 4 with 5s TTL

  ⏰ t=0s — holder=user_lazy, TTL=5s
  ⏰ t=1s — holder=user_lazy, TTL=4s
  ⏰ t=2s — holder=user_lazy, TTL=3s
  ⏰ t=3s — holder=user_lazy, TTL=2s
  ⏰ t=4s — holder=user_lazy, TTL=1s
  ⏰ t=5s — 🔓 LOCK EXPIRED! Ticket is available again!

  User 'fast' tries to reserve: ✅ Got it!


0

## 🗺️ Seat Map with Reservations

Now let's update the seat map rendering to show reserved tickets. The Event Service queries Redis for reserved ticket IDs so the client can display them as unavailable.

In [7]:
def get_event_with_reservations(event_id: int) -> dict:
    """
    Enhanced Event Service: combines DB ticket status + Redis reservation status.
    Returns three states per seat: available, reserved, sold.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Fetch event details (same as Lab 1)
    cur.execute("""
        SELECT e.name, v.name AS venue_name, v.seat_map
        FROM events e JOIN venues v ON e.venue_id = v.id
        WHERE e.id = %s
    """, (event_id,))
    event = cur.fetchone()

    # Fetch tickets
    cur.execute("""
        SELECT id, section, row_label, seat_number, price, status
        FROM tickets WHERE event_id = %s
        ORDER BY section, row_label, seat_number
    """, (event_id,))
    tickets = cur.fetchall()

    cur.close()
    conn.close()

    # Check Redis for reserved tickets
    reserved_ids = redis_client.smembers(f"event:{event_id}:reserved")

    # Merge: if ticket is 'available' in DB but reserved in Redis → show as 'reserved'
    for t in tickets:
        if t["status"] == "available" and str(t["id"]) in reserved_ids:
            # Verify the Redis lock actually still exists (TTL might have expired)
            if redis_client.exists(f"ticket:{t['id']}"):
                t["status"] = "reserved"
            else:
                # Lock expired — clean up stale entry
                redis_client.srem(f"event:{event_id}:reserved", str(t["id"]))

    return {"event_name": event["name"], "venue_name": event["venue_name"],
            "seat_map": event["seat_map"], "tickets": [dict(t) for t in tickets]}


def render_seat_map_with_reservations(data: dict):
    """Seat map with 3 states: available, reserved, sold."""
    status_map = {}
    for t in data["tickets"]:
        status_map[(t["section"], t["row_label"], t["seat_number"])] = t["status"]

    print(f"🗺️  {data['venue_name']} — {data['event_name']}\n")
    for section in data["seat_map"]["sections"]:
        section_name = section["name"]
        print(f"  ┌{'─' * 45}┐")
        print(f"  │  Section: {section_name:<33}│")
        print(f"  ├{'─' * 45}┤")
        for row in section["rows"]:
            seats = ""
            for s in range(1, row["seats"] + 1):
                st = status_map.get((section_name, row["label"], s), "available")
                if st == "sold":    seats += "● "
                elif st == "reserved": seats += "◉ "
                else:               seats += "○ "
            print(f"  │  Row {row['label']}: {seats.strip():<35}│")
        print(f"  └{'─' * 45}┘\n")
    print("  Legend: ○ available  ◉ reserved  ● sold")

# Reserve a few tickets so we can see them on the map
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id FROM tickets WHERE event_id = 1 AND status = 'available' ORDER BY id LIMIT 4")
demo_ids = [r["id"] for r in cur.fetchall()]
cur.close()
conn.close()

for tid in demo_ids:
    redis_client.set(f"ticket:{tid}", "demo_user", nx=True, ex=60)
    redis_client.sadd("event:1:reserved", str(tid))

print(f"📌 Reserved tickets {demo_ids} for demo\n")

# Render the seat map
data = get_event_with_reservations(event_id=1)
render_seat_map_with_reservations(data)

📌 Reserved tickets [4, 5, 7, 8] for demo

🗺️  Madison Square Garden — The Eras Tour - NYC

  ┌─────────────────────────────────────────────┐
  │  Section: FLOOR                            │
  ├─────────────────────────────────────────────┤
  │  Row A: ○ ○ ○ ◉ ◉ ● ◉ ◉ ● ○                │
  │  Row B: ○ ○ ○ ○ ○ ○ ○ ○ ○ ○                │
  │  Row C: ○ ○ ● ○ ○ ○ ○ ○ ○ ●                │
  └─────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────┐
  │  Section: LOWER                            │
  ├─────────────────────────────────────────────┤
  │  Row A: ○ ○ ○ ○ ● ○ ○ ● ● ○ ○ ● ○ ○ ○      │
  │  Row B: ○ ○ ● ○ ○ ○ ● ○ ○ ○ ○ ○ ○ ○ ○      │
  │  Row C: ○ ● ○ ○ ○ ● ○ ● ● ○ ○ ○ ○ ● ●      │
  └─────────────────────────────────────────────┘

  ┌─────────────────────────────────────────────┐
  │  Section: UPPER                            │
  ├─────────────────────────────────────────────┤
  │  Row A: ○ ● ○ ○ ○ ○ ○ ○ ○ ○ ○ ● ○ ● ● ○ ○ ○ ○ ○│
  │  Row B: ○

## 🧹 Cleanup

In [8]:
# Reset everything
conn = get_connection()
cur = conn.cursor()
cur.execute("DELETE FROM booking_tickets")
cur.execute("DELETE FROM bookings")
cur.execute("UPDATE tickets SET status = 'available' WHERE status = 'booked'")
conn.commit()
cur.close()
conn.close()

# Flush all Redis keys for this demo
redis_client.flushdb()
print("✅ PostgreSQL and Redis cleaned up.")

✅ PostgreSQL and Redis cleaned up.


## 🤔 Edge Cases Worth Discussing in an Interview

### 1. TTL expires during payment
User A's lock expires at minute 10, but Stripe confirms at minute 11. User B grabbed the lock in between.  
**Fix:** The DB transaction uses `SELECT ... FOR UPDATE` as a safety net — only one write succeeds. Losing user gets auto-refunded. Set TTL generously and **extend it** when payment is initiated.

### 2. Redis goes down
**Impact:** Degrades to Lab 3 behavior — no reservations, but `SELECT ... FOR UPDATE` still prevents double bookings. UX suffers (users can lose tickets while paying), but **correctness is preserved**.

### 3. Multi-ticket locking
User selects 4 seats. Lock acquisition is sequential per ticket. If ticket 3 fails, we release tickets 1 and 2.  
**Optimization:** Use a Redis Lua script to make multi-lock atomic (if tickets hash to the same node).

### 4. Stripe webhook idempotency
Stripe can retry webhooks on failure. The handler must be idempotent:
- Check booking status before updating — if already `confirmed`, skip
- Use `bookingId` as the idempotency key

### 5. Stale reserved set entries
When a Redis lock expires, the `event:{id}:reserved` set entry remains. The read path (seat map) handles this by checking `EXISTS ticket:{id}` and cleaning up stale entries. A periodic sweep can also clean the set.

## ✅ Summary: All 4 Approaches Compared

| Approach | Lock Mechanism | Expiration | Pros | Cons |
|----------|---------------|------------|------|------|
| ❌ **Long-running DB lock** | `SELECT FOR UPDATE` held open | Transaction commit/rollback | Simple | Holds connections, deadlock risk, doesn't scale |
| 🟡 **Status + cron** | `reserved` status column | Cron job sweeps expired rows | Short transactions, visible state | Cron delay, cron reliability, stale data |
| 🟢 **Inline expiration** | `reserved` + `reserved_until` | Checked inline in every reserve query | No cron dependency, self-healing | Dirty DB data, extra filter on reads |
| 🟢 **Redis distributed lock** | `SET NX EX` | Automatic TTL expiration | Fastest, auto-expiry, scales independently | Extra infra (Redis), read-path complexity |

**For a mid-level interview:** the inline expiration approach (Great 1) is a strong answer.  
**For a senior/staff interview:** discuss Redis distributed locks (Great 2), call out the trade-offs, and mention fallback behavior when Redis is down.

**Key insight:** Redis handles the ephemeral reservation state (microsecond lock acquisition, automatic TTL expiry). PostgreSQL handles the durable booking state (ACID transactions, source of truth). Each tool does what it's best at.